# Chunking Strategy Evaluation for RAG Pipeline

This notebook compares different text chunking strategies to evaluate their impact on a RAG (Retrieval-Augmented Generation) pipeline. We will use a local HTML file as the data source and the RAGAS framework to assess the performance of each strategy.

## 1. Install Dependencies

This cell installs the necessary packages for the experiment

In [ ]:
%pip install ragas langchain-experimental matplotlib seaborn jupyterlab notebook ipywidgets ipykernel pandas faiss-cpu langchain-community rapidfuzz openai

## 2. Imports and Setup

In [ ]:
# Standard library imports
import re
import sys
from pathlib import Path

# Third-party imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup
from IPython.display import display
from datasets import Dataset

# LangChain imports
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS

# RAGAS imports
from ragas import evaluate
from ragas.testset import TestsetGenerator
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
# Note: Using deprecated metrics API which is compatible with evaluate() in RAGAS 0.4.x
# The collections API uses different metric classes that are not yet compatible with evaluate()
from ragas.metrics import (
    context_precision,
    context_recall,
    faithfulness,
    answer_relevancy,
)

# OpenAI client for RAGAS (Ollama provides OpenAI-compatible API)
from openai import OpenAI

# Local imports - add project roots to sys.path
project_root = Path.cwd().parent.parent  # /home/user/documents/python/localllm
sys.path.insert(0, str(project_root / "rag" / "src"))  # for rag/src
sys.path.insert(0, str(project_root / "cli" / "src"))  # for cli/src

from rag.config import settings as rag_settings
from cli.config import settings as cli_settings

## 3. Load Data from Local HTML File

In [ ]:
# Configuration
DATA_FILE_PATH = Path("data/scraped_page.html")

def load_html_document(file_path: Path) -> Document:
    """Load and clean HTML document."""
    if not file_path.exists():
        raise FileNotFoundError(f"Data file not found: {file_path}")
    
    with open(file_path, "r", encoding="utf-8") as f:
        html_content = f.read()
    
    soup = BeautifulSoup(html_content, "html.parser")
    raw_text = soup.get_text()
    cleaned_text = re.sub(r'\\s+', ' ', raw_text).strip()
    
    return Document(page_content=cleaned_text, metadata={"source": str(file_path)})

# Load the document
documents = [load_html_document(DATA_FILE_PATH)]
print(f"Loaded and cleaned document from {DATA_FILE_PATH}")
print(f"Document length: {len(documents[0].page_content)} characters")

## 4. Define Chunking Strategies

In [ ]:
# Initialize embeddings model (shared across chunking and RAG)
ollama_embeddings = OllamaEmbeddings(
    base_url=rag_settings.ollama_base_url,
    model=rag_settings.rag_ollama_model
)

# Define chunking strategies with their configurations
CHUNKING_CONFIGS = {
    # Small chunks - high granularity, more chunks
    "Recursive (256/50)": RecursiveCharacterTextSplitter(chunk_size=256, chunk_overlap=50),
    "Recursive (512/100)": RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=100),
    
    # Medium chunks - balanced approach
    "Recursive (1024/200)": RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=200),
    "Recursive (1536/300)": RecursiveCharacterTextSplitter(chunk_size=1536, chunk_overlap=300),
    
    # Large chunks - less granularity, fewer chunks
    "Recursive (2048/400)": RecursiveCharacterTextSplitter(chunk_size=2048, chunk_overlap=400),
    "Recursive (4096/800)": RecursiveCharacterTextSplitter(chunk_size=4096, chunk_overlap=800),
    
    # Different overlap ratios (20% vs 10%)
    "Recursive (1024/100)": RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=100),
    "Recursive (1024/300)": RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=300),
    
    # Character-based splitters (no semantic awareness)
    "Character (1000/200)": CharacterTextSplitter(chunk_size=1000, chunk_overlap=200),
    "Character (2000/400)": CharacterTextSplitter(chunk_size=2000, chunk_overlap=400),
    
    # Semantic chunking - uses embeddings to find natural breaks
    "Semantic (Default)": SemanticChunker(ollama_embeddings),
    "Semantic (High Threshold)": SemanticChunker(
        ollama_embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=85
    ),
    "Semantic (Low Threshold)": SemanticChunker(
        ollama_embeddings,
        breakpoint_threshold_type="percentile",
        breakpoint_threshold_amount=60
    ),
}

# Apply chunking strategies
chunked_docs = {}
chunk_stats = []

for name, splitter in CHUNKING_CONFIGS.items():
    chunks = splitter.split_documents(documents)
    chunked_docs[name] = chunks
    
    # Calculate statistics
    chunk_lengths = [len(chunk.page_content) for chunk in chunks]
    stats = {
        "strategy": name,
        "num_chunks": len(chunks),
        "avg_chunk_size": sum(chunk_lengths) / len(chunk_lengths) if chunk_lengths else 0,
        "min_chunk_size": min(chunk_lengths) if chunk_lengths else 0,
        "max_chunk_size": max(chunk_lengths) if chunk_lengths else 0,
    }
    chunk_stats.append(stats)
    
    print(f"{name:30s} - {len(chunks):3d} chunks | avg: {stats['avg_chunk_size']:6.0f} chars | min: {stats['min_chunk_size']:4d} | max: {stats['max_chunk_size']:5d}")

# Display summary table
print("\n=== Chunking Strategy Comparison ===")
chunk_stats_df = pd.DataFrame(chunk_stats)
display(chunk_stats_df)

## 5. Generate Test Set

We will generate a small test set of questions and answers from our single document.

In [ ]:
# Configuration
TESTSET_SIZE = 5  # Number of test samples to generate

print(f"Using chat model: {cli_settings.cli_ollama_model}")
print(f"(Embedding model for vector store: {rag_settings.rag_ollama_model})")

# Initialize LLM for test set generation
generator_llm = ChatOllama(model=cli_settings.cli_ollama_model)

# Create test set generator (reuses embeddings from Step 4)
generator = TestsetGenerator.from_langchain(
    llm=generator_llm,
    embedding_model=ollama_embeddings
)

# Generate test set
testset = generator.generate_with_langchain_docs(documents, testset_size=TESTSET_SIZE)
test_df = testset.to_pandas()

print(f"\nGenerated {len(testset)} test questions")
print(f"Available columns: {list(test_df.columns)}")
display(test_df.head())

## 6. Define RAG Chain and Run Evaluation

Now we will loop through each chunking strategy, build a RAG chain with an in-memory vector store (FAISS), and evaluate its performance using RAGAS.

In [ ]:
def create_rag_chain(retriever, llm):
    """Create a RAG chain with the given retriever and LLM."""
    template = """Answer the question based only on the following context:\n
{context}\n
\n
Question: {question}"""
    prompt = PromptTemplate.from_template(template)
    
    return (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

def evaluate_strategy(name, chunks, questions, ground_truths, embeddings, llm, evaluator_llm, evaluator_embeddings):
    """Evaluate a single chunking strategy."""
    print(f"\nEvaluating strategy: {name}")
    
    vectorstore = FAISS.from_documents(chunks, embeddings)
    retriever = vectorstore.as_retriever()
    
    rag_chain = create_rag_chain(retriever, llm)
    
    answers = []
    contexts = []
    
    for question in questions:
        ans = rag_chain.invoke(question)
        answers.append(ans)
        
        retrieved_docs = retriever.invoke(question)
        contexts.append([doc.page_content for doc in retrieved_docs])
    
    # Create dataset for evaluation
    response_dataset = Dataset.from_dict({
        "user_input": questions,
        "response": answers,
        "retrieved_contexts": contexts,
        "reference": [gt[0] for gt in ground_truths]
    })
    
    # Run RAGAS evaluation
    result = evaluate(
        response_dataset,
        metrics=[
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy,
        ],
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        raise_exceptions=False
    )
    
    # Add strategy name to results
    result_dict = dict(result)
    result_dict['strategy'] = name
    
    return result_dict

# Prepare test data - extract from testset samples
questions = [item.eval_sample.user_input for item in testset]
ground_truths = [[item.eval_sample.reference] for item in testset]

# Initialize LLM for RAG chain (generates answers)
chat_llm = ChatOllama(model=cli_settings.cli_ollama_model)

ollama_client = OpenAI(
    base_url=f"{rag_settings.ollama_base_url}/v1",
    api_key="ollama"  # Ollama doesn't require a real API key
)
evaluator_llm = llm_factory(
    model=cli_settings.cli_ollama_model,
    client=ollama_client
)
print(f"RAGAS evaluator LLM initialized: {cli_settings.cli_ollama_model}")

evaluator_embeddings = embedding_factory(
    "openai",
    model=rag_settings.rag_ollama_model,
    client=ollama_client
)
print(f"RAGAS evaluator embeddings initialized: {rag_settings.rag_ollama_model}")

# Evaluate all chunking strategies
results_list = []
for name, chunks in chunked_docs.items():
    result = evaluate_strategy(
        name=name,
        chunks=chunks,
        questions=questions,
        ground_truths=ground_truths,
        embeddings=ollama_embeddings,
        llm=chat_llm,
        evaluator_llm=evaluator_llm,
        evaluator_embeddings=evaluator_embeddings
    )
    results_list.append(result)

results_df = pd.DataFrame(results_list)
display(results_df)


## 7. Visualize Results

In [ ]:
# Display summary statistics
print("=== Summary Statistics ===")
summary_df = results_df.set_index('strategy')
display(summary_df.describe().round(3))

# Display best strategy per metric
print("\n=== Best Strategy per Metric ===")
for metric in ['context_precision', 'context_recall', 'faithfulness', 'answer_relevancy']:
    if metric in summary_df.columns:
        best_strategy = summary_df[metric].idxmax()
        best_score = summary_df[metric].max()
        print(f"{metric}: {best_strategy} ({best_score:.3f})")

# Create visualization
results_df_melted = results_df.melt(id_vars='strategy', var_name='metric', value_name='score')

plt.figure(figsize=(14, 8))
sns.barplot(data=results_df_melted, x='strategy', y='score', hue='metric')
plt.title('Chunking Strategy Performance Comparison', fontsize=14, fontweight='bold')
plt.ylabel('Score', fontsize=12)
plt.xlabel('Strategy', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.legend(title='Metric', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Display overall ranking
print("\n=== Overall Ranking (by average score) ===")
summary_df['average'] = summary_df.mean(axis=1)
ranking = summary_df['average'].sort_values(ascending=False)
for i, (strategy, score) in enumerate(ranking.items(), 1):
    print(f"{i}. {strategy}: {score:.3f}")